
# Quaia G20.5 completeness and purity vs SDSS

This notebook estimates **completeness** and **purity** for `quaia_G20.5` relative to SDSS spectroscopy.

Definitions used here:

- **Completeness** (SDSS QSO -> Quaia): fraction of SDSS spectroscopic QSOs that have a Quaia counterpart within 1 arcsec.
- **Purity** (Quaia -> SDSS class): fraction of Quaia sources with SDSS spectra that are classified as `QSO`.

For all proportions, we report **Jeffreys binomial 95% intervals** (Beta posterior with prior Beta(0.5, 0.5)).


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.stats import beta

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True


In [2]:

# Paths
fn_quaia = Path('../data/quaia_G20.5.fits')
fn_sdss_specall = Path('../data/sdss_specall.csv')
fn_sdss_dr16q = Path('../data/dr16q_prop_May01_2024.fits.gz')

# Existing precomputed match made elsewhere in this repo (Quaia <-> DR16Q)
fn_match_dr16q = Path('../data/match_quaia_G20.5_sdss_dr16q_prop.fits')

# Optional cache we create for Quaia <-> SDSS specall matching
fn_match_specall_cache = Path('../data/match_quaia_G20.5_sdss_specall_1as.fits')

for fn in [fn_quaia, fn_sdss_specall, fn_sdss_dr16q, fn_match_dr16q]:
    assert fn.exists(), f'Missing required file: {fn}'


In [3]:

# Load main catalogs
quaia = Table.read(fn_quaia)
sdss_specall = Table.from_pandas(pd.read_csv(fn_sdss_specall))
sdss_dr16q = Table.read(fn_sdss_dr16q, hdu=1)

# Ensure consistent class dtype and clean obvious whitespace
if 'class' in sdss_specall.colnames:
    sdss_specall['class'] = np.char.strip(np.asarray(sdss_specall['class']).astype(str))

print(f'Quaia G20.5: {len(quaia):,}')
print(f'SDSS specall: {len(sdss_specall):,}')
print(f'SDSS DR16Q: {len(sdss_dr16q):,}')


Quaia G20.5: 1,295,502
SDSS specall: 5,112,724
SDSS DR16Q: 750,414


In [4]:

def jeffreys_interval(k, n, alpha=0.05):
    """Return posterior mean and equal-tail Jeffreys interval for binomial proportion."""
    if n <= 0:
        return np.nan, np.nan, np.nan
    a = k + 0.5
    b = n - k + 0.5
    mean = a / (a + b)
    lo = beta.ppf(alpha / 2, a, b)
    hi = beta.ppf(1 - alpha / 2, a, b)
    return mean, lo, hi


def summarize_fraction(k, n, label='fraction'):
    mean, lo, hi = jeffreys_interval(k, n)
    print(f"{label}: {k}/{n} = {k/n:.4f}")
    print(f"  Jeffreys mean = {mean:.4f}, 95% CI = [{lo:.4f}, {hi:.4f}]")


In [5]:

# --- Completeness using precomputed Quaia <-> SDSS DR16Q matches ---
match_dr16q = Table.read(fn_match_dr16q)

# Match file stores one matched SDSS DR16Q object per row (OBJID in sdss_objid)
n_match_dr16q = len(match_dr16q)
n_sdss_dr16q = len(sdss_dr16q)

print(f'Matched Quaia<->DR16Q rows: {n_match_dr16q:,}')
print(f'Total DR16Q rows: {n_sdss_dr16q:,}')

summarize_fraction(n_match_dr16q, n_sdss_dr16q, label='Completeness (DR16Q recovered in Quaia)')


Matched Quaia<->DR16Q rows: 294,591
Total DR16Q rows: 750,414
Completeness (DR16Q recovered in Quaia): 294591/750414 = 0.3926
  Jeffreys mean = 0.3926, 95% CI = [0.3915, 0.3937]


In [6]:

# --- Purity: Quaia -> SDSS spectroscopic class ---
# We match each Quaia source to its nearest SDSS specall source and keep matches within 1 arcsec.
# Cache the result for reproducibility and to avoid repeating an expensive sky match.

def match_nearest_within(ra1_deg, dec1_deg, ra2_deg, dec2_deg, max_sep_arcsec=1.0):
    c1 = SkyCoord(ra=np.asarray(ra1_deg) * u.deg, dec=np.asarray(dec1_deg) * u.deg, frame='icrs')
    c2 = SkyCoord(ra=np.asarray(ra2_deg) * u.deg, dec=np.asarray(dec2_deg) * u.deg, frame='icrs')
    idx2, sep2d, _ = c1.match_to_catalog_sky(c2)
    keep = sep2d < (max_sep_arcsec * u.arcsec)
    idx1 = np.where(keep)[0]
    return idx1, idx2[keep], sep2d[keep].arcsec


if fn_match_specall_cache.exists():
    match_specall = Table.read(fn_match_specall_cache)
else:
    i_quaia, i_sdss, sep_arcsec = match_nearest_within(
        quaia['ra'], quaia['dec'], sdss_specall['ra'], sdss_specall['dec'], max_sep_arcsec=1.0
    )

    match_specall = Table()
    match_specall['source_id'] = quaia['source_id'][i_quaia]
    match_specall['sdss_index'] = i_sdss
    match_specall['sep_arcsec'] = sep_arcsec
    match_specall['sdss_class'] = sdss_specall['class'][i_sdss]
    match_specall['sdss_zWarning'] = sdss_specall['zWarning'][i_sdss]
    match_specall['sdss_sciencePrimary'] = sdss_specall['sciencePrimary'][i_sdss]
    match_specall.write(fn_match_specall_cache, overwrite=True)

print(f'Quaia objects with SDSS specall match (<=1 arcsec): {len(match_specall):,}')


Quaia objects with SDSS specall match (<=1 arcsec): 303,203


In [7]:

# Global purity among Quaia objects that have SDSS spectra
is_qso = np.asarray(match_specall['sdss_class']) == 'QSO'

k = int(np.sum(is_qso))
n = len(match_specall)
summarize_fraction(k, n, label='Purity (all SDSS classes, any zWarning)')

# A stricter estimate using only zWarning==0 and sciencePrimary==1
zw0 = np.asarray(match_specall['sdss_zWarning']) == 0
sp1 = np.asarray(match_specall['sdss_sciencePrimary']) == 1
good = zw0 & sp1

k_good = int(np.sum(is_qso & good))
n_good = int(np.sum(good))
summarize_fraction(k_good, n_good, label='Purity (zWarning==0 & sciencePrimary==1)')


Purity (all SDSS classes, any zWarning): 300806/303203 = 0.9921
  Jeffreys mean = 0.9921, 95% CI = [0.9918, 0.9924]
Purity (zWarning==0 & sciencePrimary==1): 293901/294693 = 0.9973
  Jeffreys mean = 0.9973, 95% CI = [0.9971, 0.9975]


In [8]:

# Completeness and purity as a function of Quaia G magnitude
# (all use Jeffreys intervals per bin)

g = np.asarray(quaia['phot_g_mean_mag'])

# map source_id -> G for matched rows
sid_to_g = pd.Series(g, index=np.asarray(quaia['source_id']))
g_match_dr16q = sid_to_g.reindex(np.asarray(match_dr16q['source_id'])).to_numpy()

g_match_specall = sid_to_g.reindex(np.asarray(match_specall['source_id'])).to_numpy()
class_match_specall = np.asarray(match_specall['sdss_class'])

bins = np.arange(16.0, 20.6, 0.3)
centers = 0.5 * (bins[:-1] + bins[1:])

# Completeness vs G: among DR16Q-matched Quaia entries (numerator proxy) over all Quaia in bin is not valid.
# Instead use SDSS->Quaia matching in next cell for a true completeness-vs-mag (SDSS-frame).
# Here we only show purity-vs-G in Quaia-frame.

pur_mean, pur_lo, pur_hi = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (g_match_specall >= lo) & (g_match_specall < hi)
    nbin = int(np.sum(m))
    kbin = int(np.sum(class_match_specall[m] == 'QSO'))
    mean, ci_lo, ci_hi = jeffreys_interval(kbin, nbin)
    pur_mean.append(mean)
    pur_lo.append(ci_lo)
    pur_hi.append(ci_hi)

pur_mean = np.asarray(pur_mean)
pur_lo = np.asarray(pur_lo)
pur_hi = np.asarray(pur_hi)

plt.figure()
plt.errorbar(centers, pur_mean, yerr=[pur_mean - pur_lo, pur_hi - pur_mean], fmt='o', capsize=3)
plt.ylim(0, 1.02)
plt.xlabel('Quaia phot_g_mean_mag')
plt.ylabel('Purity (P[QSO | SDSS match])')
plt.title('Quaia purity vs G (SDSS-classified subset)')
plt.show()


ValueError: Big-endian buffer not supported on little-endian compiler

In [ ]:

# True completeness vs SDSS redshift: SDSS DR16Q frame
# We use the saved matched OBJIDs to mark recovered SDSS quasars.

objid_key = 'OBJID'
z_key = 'Z_DR16Q'

sdss_objid = np.asarray(sdss_dr16q[objid_key])
matched_objid = set(np.asarray(match_dr16q['sdss_objid']))
recovered = np.array([oid in matched_objid for oid in sdss_objid], dtype=bool)

z = np.asarray(sdss_dr16q[z_key], dtype=float)
valid = np.isfinite(z)

z_bins = np.arange(0.0, 6.2, 0.3)
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])

comp_mean, comp_lo, comp_hi = [], [], []
for lo, hi in zip(z_bins[:-1], z_bins[1:]):
    m = valid & (z >= lo) & (z < hi)
    nbin = int(np.sum(m))
    kbin = int(np.sum(recovered[m]))
    mean, ci_lo, ci_hi = jeffreys_interval(kbin, nbin)
    comp_mean.append(mean)
    comp_lo.append(ci_lo)
    comp_hi.append(ci_hi)

comp_mean = np.asarray(comp_mean)
comp_lo = np.asarray(comp_lo)
comp_hi = np.asarray(comp_hi)

plt.figure()
plt.errorbar(z_centers, comp_mean, yerr=[comp_mean - comp_lo, comp_hi - comp_mean], fmt='o', capsize=3)
plt.ylim(0, 1.02)
plt.xlabel('SDSS DR16Q redshift')
plt.ylabel('Completeness (P[matched in Quaia | SDSS QSO])')
plt.title('Quaia completeness vs SDSS redshift')
plt.show()



## Notes and caveats

- Completeness is computed in the SDSS DR16Q frame and includes footprint/selection differences between SDSS and Gaia/Quaia.
- Purity is computed only on the subset of Quaia objects with SDSS spectroscopy; this can be biased relative to the full all-sky Quaia catalog.
- If desired, this can be extended to conditional estimates (e.g., in common sky footprint, by Galactic latitude, or with additional quality cuts).
